In [1]:
from datasets import load_dataset

ds = load_dataset("Genius-Society/Pima")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [2]:
!pip install pennylane datasets

import pennylane as qml
from pennylane import numpy as np
# Extract features and labels into arrays
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'], s['SkinThickness'],
               s['Insulin'], s['BMI'], s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']])
y = np.array([s['Outcome'] for s in ds['train']])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 60.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [3]:
# ============================================================
# ÉTAPE 1 : Normalisation des données (important pour les angles quantiques)
# ============================================================

X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_range = np.where(X_max - X_min == 0, 1, X_max - X_min)
X_norm = (X - X_min) / X_range  # mise à l’échelle entre 0 et 1
print(X_norm)

[[0.23529412 0.81935484 0.45652174 ... 0.61650485 0.16873662 0.09803922]
 [0.17647059 0.55483871 0.36956522 ... 0.11893204 0.09807281 0.01960784]
 [0.23529412 0.70967742 0.45652174 ... 0.31796117 0.10835118 0.31372549]
 ...
 [0.35294118 0.94193548 0.67391304 ... 0.41990291 0.08265525 0.88235294]
 [0.58823529 0.36774194 0.5        ... 0.35679612 0.03683084 0.82352941]
 [0.11764706 0.19354839 0.45652174 ... 0.34605494 0.00728051 0.01960784]]


In [4]:
# ============================================================
# ÉTAPE 2 : Réduction de dimension (pour adapter au nombre de qubits)
# ============================================================

from sklearn.decomposition import PCA

n_qubits = 3  # tu peux augmenter à 4–5 si ta machine le supporte
pca = PCA(n_components=n_qubits, random_state=42)
X_reduced = pca.fit_transform(X_norm)
# ============================================================
# ÉTAPE 3 : Définir l’appareil quantique et la feature map
# ============================================================

import pennylane as qml
from pennylane import numpy as np

dev = qml.device("default.qubit", wires=n_qubits)

def feature_map(x):
    """
    Circuit d’encodage (feature map) :
    - crée une superposition avec H
    - encode les features dans des rotations RZ
    - entangle les qubits avec CZ
    - ajoute des RY pour enrichir les relations non-linéaires
    """
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    for i in range(n_qubits):
        qml.RZ(np.pi * float(x[i]), wires=i)
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i+1])
    for i in range(n_qubits):
        qml.RY(np.pi * float(x[i]), wires=i)
@qml.qnode(dev)
def feature_state(x):
    """Retourne l’état quantique (vecteur d’amplitudes) correspondant à un échantillon x"""
    feature_map(x)
    return qml.state()
# ============================================================
# ÉTAPE 4 : Calcul de la matrice de noyau quantique
# ============================================================
def compute_kernel_matrix(X):
    """
    Calcule la matrice de noyau K[i,j] = |<φ(x_i)|φ(x_j)>|²
    où φ(x) est l’état quantique correspondant à x.
    """
    n = X.shape[0]
    K = np.zeros((n, n))
    states = [feature_state(x) for x in X]  # Calcul une fois les états
    for i in range(n):
        for j in range(i, n):
            overlap = np.vdot(states[i], states[j])
            value = np.abs(overlap) ** 2
            K[i, j] = value
            K[j, i] = value
    return K

print("Calcul de la matrice de noyau quantique en cours...")
K = compute_kernel_matrix(X_reduced)
print("Matrice de noyau calculée avec succès.")




Calcul de la matrice de noyau quantique en cours...
Matrice de noyau calculée avec succès.


In [5]:
# ============================================================
# ÉTAPE 5 : Afficher un aperçu de la matrice de noyau
# ============================================================

print("\nExtrait de la matrice de noyau (8x8) :")
print(K[:8, :8])


Extrait de la matrice de noyau (8x8) :
[[1.         0.35872596 0.64068849 0.39785982 0.5518045  0.3768552
  0.24655208 0.18307537]
 [0.35872596 1.         0.54397765 0.01104166 0.11100646 0.72892489
  0.96549771 0.29144838]
 [0.64068849 0.54397765 1.         0.40783162 0.14649672 0.36190216
  0.40305375 0.65191807]
 [0.39785982 0.01104166 0.40783162 1.         0.13892037 0.00615255
  0.00169858 0.16013577]
 [0.5518045  0.11100646 0.14649672 0.13892037 1.         0.24706039
  0.10035874 0.03696706]
 [0.3768552  0.72892489 0.36190216 0.00615255 0.24706039 1.
  0.71906091 0.20069351]
 [0.24655208 0.96549771 0.40305375 0.00169858 0.10035874 0.71906091
  1.         0.21036405]
 [0.18307537 0.29144838 0.65191807 0.16013577 0.03696706 0.20069351
  0.21036405 1.        ]]


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Séparation des indices pour le train et le test
# On utilise les indices car la matrice K est globale (N x N)
indices = np.arange(len(X_reduced))
idx_train, idx_test, y_train, y_test = train_test_split(indices, y, test_size=0.2, random_state=42)

# 2. Extraction des sous-matrices de noyau
# Pour l'entraînement, on regarde la similarité des données de train entre elles
X_train_kernel = K[np.ix_(idx_train, idx_train)]
# Pour le test, on regarde la similarité des données de test par rapport aux données de train
X_test_kernel = K[np.ix_(idx_test, idx_train)]

# 3. Entraînement de l'Arbre de Décision
clf_kernel = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_kernel.fit(X_train_kernel, y_train)

# 4. Prédiction
y_pred_k = clf_kernel.predict(X_test_kernel)
print(f"Précision avec le Quantum Kernel (3 qubits) : {accuracy_score(y_test, y_pred_k):.2%}")

Précision avec le Quantum Kernel (3 qubits) : 76.42%
